# HOGformer baseline (Setting I) on Colab
**Runtime:** Runtime -> Change runtime type -> **Python 3**, hardware accelerator **GPU** (A100 if you have Colab Pro; otherwise T4 / L4).
Run the cells top to bottom. If Colab disconnects, run **all cells again except the speed test**: training resumes automatically from Google Drive.

In [ ]:
# 1) Confirm a GPU is attached
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
# 2) Mount Google Drive (checkpoints and logs are stored here so they survive disconnects)
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/hogformer_runs'
!mkdir -p {RUNS}/experiments_baseline

In [ ]:
# 3) Get the code and install dependencies (torch is already on Colab)
%cd /content
!git clone -b main https://github.com/adityaj1706/HOGformer_DLCV.git
%cd /content/HOGformer_DLCV
!pip install -q -r requirements.txt

In [ ]:
# 4) Download and arrange the dataset on the fast local disk (~15-25 min, ~15 GB)
!bash scripts/download_data.sh /content/data

In [ ]:
# 5) Keep experiments/ (checkpoints, training states, logs) on Drive so training can resume
%cd /content/HOGformer_DLCV/settingI
!rm -rf experiments
!ln -s {RUNS}/experiments_baseline experiments
!ls -la experiments

## Speed test (do this once, **skip it when resuming**)
Runs 100 iterations to measure it/s on your GPU, then deletes the test run. Use the number to pick `total_iter`:
`hours = total_iter / it_per_s / 3600`.

In [ ]:
%cd /content/HOGformer_DLCV/settingI
!sed -E -e 's/^name: .*/name: speedtest/' \
     -e 's/total_iter: [0-9]+/total_iter: 100/' \
     -e 's/iters: \[[0-9]+\]/iters: [100]/' \
     -e 's/periods: \[[0-9, ]+\]/periods: [50, 50]/' \
     -e 's/val_freq: .*/val_freq: !!float 1e9/' \
     -e 's/save_checkpoint_freq: .*/save_checkpoint_freq: !!float 1e9/' \
     Allweather/Options/Allweather_HOGformer_baseline.yml > /content/speedtest.yml
!grep -E "^name|total_iter|iters:|periods|gt_sizes|mini_batch" /content/speedtest.yml
!python basicsr/train.py -opt /content/speedtest.yml --launcher none 2>&1 | grep -E "iter: +[0-9]+0,|End of training|Error|Traceback"
!rm -rf experiments/speedtest tb_logger/speedtest


Look at `time (data)` in the log: the **first** number is seconds per iteration.
Training time in hours = `total_iter x seconds_per_iter / 3600`.
If the time is too long, lower `total_iter`, `iters` and `periods` in
`settingI/Allweather/Options/Allweather_HOGformer_baseline.yml` (keep periods at about 1/3 and 2/3 of the total)
**before the first real run**, and use identical values for the proposed model.

In [ ]:
# 6) TRAIN the baseline (re-running this cell resumes from the latest saved state on Drive)
%cd /content/HOGformer_DLCV/settingI
!python basicsr/train.py -opt Allweather/Options/Allweather_HOGformer_baseline.yml --launcher none 2>&1 | grep -E "iter: +[0-9]*00,|Validation|Resuming|End of training|Error|Traceback"

In [ ]:
# 7) Where things are saved + final numbers so far
!ls experiments/Allweather_HOGformer_baseline/models | tail -3
!grep -h "Validation" experiments/Allweather_HOGformer_baseline/train_*.log | tail -6